In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_STATE = 42
DATA_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]


In [2]:
train_full = pd.read_csv(DATA_DIR / "train.csv")
train_full["any_label"] = train_full[LABEL_COLS].max(axis=1)

clean_rows = train_full[train_full["any_label"] == 0]
normal_sample = clean_rows.sample(n=10000, random_state=RANDOM_STATE)

LABEL_TARGETS = {
    "toxic": 5000,
    "obscene": 2500,
    "insult": 2500,
    "severe_toxic": 1500,
    "identity_hate": 1500,
    "threat": 1500,
}

label_samples = []
for label, target in LABEL_TARGETS.items():
    label_rows = train_full[train_full[label] == 1]
    n = min(target, len(label_rows))
    label_samples.append(label_rows.sample(n=n, random_state=RANDOM_STATE))

toxic_side = pd.concat(label_samples).drop_duplicates(subset="id")

subsample = pd.concat([toxic_side, normal_sample]).drop_duplicates(subset="id")
subsample = subsample.sample(frac=1, random_state=RANDOM_STATE)
subsample = subsample.drop(columns=["any_label"]).reset_index(drop=True)

print("Subsample size:", len(subsample))
print(subsample[LABEL_COLS].sum())


Subsample size: 19224
toxic            8794
severe_toxic     1560
obscene          6038
threat            478
insult           5748
identity_hate    1405
dtype: int64


In [3]:
n_val = int(len(subsample) * 0.1)
val_subsample = subsample.iloc[:n_val].reset_index(drop=True)
train_subsample = subsample.iloc[n_val:].reset_index(drop=True)

print("train:", len(train_subsample), " val:", len(val_subsample))

train_subsample.to_csv(PROC_DIR / "train_subsample.csv", index=False)
val_subsample.to_csv(PROC_DIR / "val_subsample.csv", index=False)


train: 17302  val: 1922


In [4]:
test_texts = pd.read_csv(DATA_DIR / "test.csv")
test_labels = pd.read_csv(DATA_DIR / "test_labels.csv")

test = test_texts.merge(test_labels, on="id")
test = test[(test[LABEL_COLS] != -1).all(axis=1)].reset_index(drop=True)

print("Usable official test rows:", len(test))
test.to_csv(PROC_DIR / "test_official_filtered.csv", index=False)
test.head()


Usable official test rows: 63978


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train = vectorizer.fit_transform(train_subsample["comment_text"])
X_val = vectorizer.transform(val_subsample["comment_text"])
X_test = vectorizer.transform(test["comment_text"])

y_train = train_subsample[LABEL_COLS].values
y_val = val_subsample[LABEL_COLS].values
y_test = test[LABEL_COLS].values

baseline_model = OneVsRestClassifier(
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
)
baseline_model.fit(X_train, y_train)
print("Baseline trained.")


Baseline trained.


In [6]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report

def evaluate(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    macro_auc = roc_auc_score(y_true, y_prob, average="macro")
    per_label_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    per_label_auc = roc_auc_score(y_true, y_prob, average=None)
    return macro_f1, macro_auc, per_label_f1, per_label_auc

test_prob = baseline_model.predict_proba(X_test)
macro_f1, macro_auc, per_label_f1, per_label_auc = evaluate(y_test, test_prob)

print(f"Baseline macro F1:  {macro_f1:.3f}")
print(f"Baseline macro AUC: {macro_auc:.3f}")

pd.DataFrame({
    "label": LABEL_COLS,
    "f1": per_label_f1,
    "roc_auc": per_label_auc,
})


Baseline macro F1:  0.446
Baseline macro AUC: 0.968


,label,f1,roc_auc
0,toxic,0.586397,0.945721
1,severe_toxic,0.252941,0.981322
2,obscene,0.644435,0.967688
3,threat,0.281046,0.981299
4,insult,0.571698,0.958274
5,identity_hate,0.342294,0.975549


In [7]:
import json

metrics_path = Path("../reports/metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)

all_metrics = {}
if metrics_path.exists():
    all_metrics = json.loads(metrics_path.read_text())

all_metrics["baseline_tfidf_logreg"] = {
    "macro_f1": float(macro_f1),
    "macro_auc": float(macro_auc),
    "per_label_f1": dict(zip(LABEL_COLS, per_label_f1.tolist())),
    "per_label_auc": dict(zip(LABEL_COLS, per_label_auc.tolist())),
}

metrics_path.write_text(json.dumps(all_metrics, indent=2))
print("Saved to", metrics_path)



np.save(PROC_DIR / "baseline_test_probs.npy", test_prob)


Saved to ../reports/metrics.json
